In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [29]:
df = pd.read_csv('car_dataset_dirty_520_rows.csv')
print(df)

        Brand  Car_Name  Year  Selling_Price  Present_Price  Kms_Driven  \
0        Ford  EcoSport  2019      1478346.0     3387994.90       32345   
1     Hyundai      Kwid  2022       225152.0      259043.00      156112   
2       Skoda    Seltos  2018       722551.0     1186149.58      102511   
3     Hyundai  EcoSport  2009       229976.0      538890.66       91037   
4         NaN      Kwid  2011      1500311.0     2228016.16      247955   
..        ...       ...   ...            ...            ...         ...   
515     Skoda     Swift  2020       620523.0     1008479.00       82223   
516  Mahindra    XUV500  2008       604993.0     1274425.13       38278   
517     Skoda     Swift  2020       925710.0     2238900.54       99344   
518      Ford       i20  2008       284896.0      345163.65       18534   
519       Kia    XUV500  2025       626316.0     1046697.85      162362   

    Fuel_Type Transmission  Owner  
0      Petrol       Manual      0  
1      Petrol       Manual 

In [2]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('car_dataset_dirty_520_rows.csv')

df.columns = df.columns.str.strip().str.lower()

price_cols = [col for col in df.columns if 'price' in col]

if not price_cols:
    raise ValueError(f"No price column found! Available columns are: {list(df.columns)}")

target_col = price_cols[0] 
print(f"Target column successfully identified as: '{target_col}'")

y = df[target_col]
X = df.drop(columns=[target_col])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Data successfully split! X_train shape: {X_train.shape}")


Target column successfully identified as: 'selling_price'
Data successfully split! X_train shape: (416, 8)


In [23]:

# Import Libraries
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load Dataset
df = pd.read_csv('car_dataset_dirty_520_rows.csv')

df.columns = df.columns.str.strip()

df.drop("Car_Name", axis=1, inplace=True, errors='ignore')

# Calculate Car Age
df["Current_Year"] = 2025

year_col = [col for col in df.columns if col.lower() == 'year'][0]
df["Car_Age"] = df["Current_Year"] - df[year_col]

# Remove unnecessary columns
df.drop([year_col, "Current_Year"], axis=1, inplace=True)

for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# Convert categorical columns into numerical values
df = pd.get_dummies(df, drop_first=True)

target_col = [col for col in df.columns if 'selling_price' in col.lower()][0]
X = df.drop(target_col, axis=1)
y = df[target_col]

# Split Dataset into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Create Random Forest Model
model = RandomForestRegressor(n_estimators=200, random_state=42)

# Train the Model
model.fit(X_train, y_train)

# Predict on Test Data
predictions = model.predict(X_test)

# Evaluate the Model
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)
print("Root Mean Squared Error (RMSE):", rmse)
print("R² Score:", r2)


# Save the Trained Model
joblib.dump(model,'car_dataset_dirty_520_rows.csv')


Root Mean Squared Error (RMSE): 5096337.783173231
R² Score: -0.12207364235756746


['car_dataset_dirty_520_rows.csv']

In [30]:
import pandas as pd
import joblib

# Load model
model = joblib.load('car_dataset_dirty_520_rows.csv')

# Create DataFrame with all required columns
data = pd.DataFrame(0, index=[0], columns=model.feature_names_in_)

# User Input
data.loc[0, "Present_Price"] = float(input("Present Price (Lakhs): "))
data.loc[0, "Kms_Driven"] = int(input("Kilometers Driven: "))
data.loc[0, "Owner"] = int(input("Previous Owners (0-3): "))
data.loc[0, "Car_Age"] = int(input("Car Age (Years): "))

brand = input("Brand: ").strip().title()
fuel = input("Fuel Type (Petrol/Diesel/CNG): ").strip().title()
trans = input("Transmission (Manual/Automatic): ").strip().title()

# Brand
brand_col = "Brand_" + brand
if brand_col in data.columns:
    data.loc[0, brand_col] = 1

# Fuel
fuel_col = "Fuel_Type_" + fuel
if fuel_col in data.columns:
    data.loc[0, fuel_col] = 1

# Transmission
if trans == "Manual" and "Transmission_Manual" in data.columns:
    data.loc[0, "Transmission_Manual"] = 1

# Predict
price = model.predict(data)

print("\nEstimated Selling Price: ₹ {:.2f} Lakhs".format(price[0]))

Present Price (Lakhs):  1200000
Kilometers Driven:  23000
Previous Owners (0-3):  1
Car Age (Years):  1
Brand:  honda
Fuel Type (Petrol/Diesel/CNG):  petrol
Transmission (Manual/Automatic):  automatic



Estimated Selling Price: ₹ 783884.73 Lakhs
